##Feature Engineering

### Step 1: Load Gold Data

In [0]:
gold_df = spark.read.table("workspace.default.gold_heart_disease")
display(gold_df)
gold_df.printSchema()

### Step 2: Create Clinically Meaningful Features

In [0]:
from pyspark.sql.functions import when

feature_df = gold_df \
    .withColumn("age_group",
        when(gold_df.age < 40, "Young")
        .when(gold_df.age < 60, "Middle_Age")
        .otherwise("Senior")
    ) \
    .withColumn("bp_risk_level",
        when(gold_df.resting_blood_pressure < 120, "Normal")
        .when(gold_df.resting_blood_pressure < 140, "Elevated")
        .otherwise("High")
    ) \
    .withColumn("cholesterol_risk_level",
        when(gold_df.serum_cholesterol < 200, "Normal")
        .when(gold_df.serum_cholesterol < 240, "Borderline_High")
        .otherwise("High")
    )

feature_df.select("age","age_group","resting_blood_pressure","bp_risk_level",
                  "serum_cholesterol","cholesterol_risk_level").show(10)


In [0]:
from pyspark.sql.functions import col

feature_df = feature_df.withColumn("patient_id", col("patient_id").cast("long"))

In [0]:
feature_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_heart_feature_details")

### Step 3: Prepare ML Dataset (Numeric Encoding)

In [0]:
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(inputCol="age_group", outputCol="age_group_index"),
    StringIndexer(inputCol="bp_risk_level", outputCol="bp_risk_index"),
    StringIndexer(inputCol="cholesterol_risk_level", outputCol="cholesterol_risk_index"),
    StringIndexer(inputCol="gender_label", outputCol="gender_index")
]

for indexer in indexers:
    feature_df = indexer.fit(feature_df).transform(feature_df)


### Step 4: Assemble Features for ML

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "age",
    "resting_blood_pressure",
    "serum_cholesterol",
    "max_heart_rate",
    "st_depression",
    "num_major_vessels",
    "age_group_index",
    "bp_risk_index",
    "cholesterol_risk_index",
    "gender_index"
]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
final_ml_df = assembler.transform(feature_df).select("patient_id","features", "heart_disease_flag")
final_ml_df.show(5)


In [0]:
final_ml_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_heart_features")


### Step 5: Train-Test Split

In [0]:
train_df, test_df = final_ml_df.randomSplit([0.8, 0.2], seed=42)


## MLflow Model Training 

### Step 1: Train Logistic Regression (Baseline Model)

In [0]:
import mlflow
import mlflow.spark
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from mlflow.models.signature import infer_signature

mlflow.set_experiment("/Shared/Heart_Disease_Risk_Prediction")

with mlflow.start_run(run_name="Logistic_Regression"):
    lr = LogisticRegression(featuresCol="features", labelCol="heart_disease_flag")
    lr_model = lr.fit(train_df)
    
    lr_predictions = lr_model.transform(train_df)
    evaluator = BinaryClassificationEvaluator(labelCol="heart_disease_flag")
    auc_lr = evaluator.evaluate(lr_model.transform(test_df))
    
    signature = infer_signature(train_df.toPandas(), lr_predictions.toPandas())
    mlflow.log_param("model_type", "Logistic Regression")
    mlflow.log_metric("AUC", auc_lr)
    mlflow.spark.log_model(lr_model, "logistic_regression_model", dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp/", signature=signature)
    
    print(f"Logistic Regression AUC: {auc_lr}")


### Step 2: Train Random Forest (Advanced Model)

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from mlflow.models.signature import infer_signature

with mlflow.start_run(run_name="Random_Forest"):
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="heart_disease_flag",
        numTrees=100,
        maxDepth=5,
        seed=42
    )
    
    rf_model = rf.fit(train_df)
    rf_predictions = rf_model.transform(train_df)
    auc_rf = evaluator.evaluate(rf_model.transform(test_df))
    
    signature = infer_signature(train_df.toPandas(), rf_predictions.toPandas())
    mlflow.log_param("model_type", "Random Forest")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 5)
    mlflow.log_metric("AUC", auc_rf)
    mlflow.spark.log_model(rf_model, "random_forest_model", dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp/", signature=signature)
    
    print(f"Random Forest AUC: {auc_rf}")


### Step 3: Feature Importance (Clinical Explainability)

In [0]:
from pyspark.sql import Row

# Get feature names (same order as VectorAssembler)
feature_names = feature_columns  # already defined earlier

# Extract feature importance from Random Forest model
importances = rf_model.featureImportances.toArray()

# Create Spark DataFrame
fi_rows = [
    Row(feature=feature_names[i], importance=float(importances[i]))
    for i in range(len(feature_names))
]

feature_importance_df = spark.createDataFrame(fi_rows)

feature_importance_df.show()

In [0]:
feature_importance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_feature_importance")

### Feature Importance Analysis

This analysis shows which clinical and engineered features most strongly influence heart disease prediction.
The model identifies:

- Structural heart indicators (number of vessels, ST depression)

- Physiological stress markers (max heart rate, blood pressure)

- Demographic risk (age, gender)

- Derived risk indices (BP, cholesterol, age group)
